# Source-image leakage audit

This notebook reconstructs the exact CASIA v2 split used in the thesis and counts how many tampered test images have their authentic source image present in the training split.


In [1]:
import random
import re
import warnings
from pathlib import Path

import numpy as np
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

print(f"Seed: {SEED}")

Seed: 42


In [2]:
import kagglehub

path = kagglehub.dataset_download("divg07/casia-20-image-tampering-detection-dataset")
DATA_ROOT = Path(path) / "CASIA2"

au_paths = sorted(p for p in (DATA_ROOT / "Au").iterdir() if p.suffix.lower() in IMG_EXTS)
tp_paths = sorted(p for p in (DATA_ROOT / "Tp").iterdir() if p.suffix.lower() in IMG_EXTS)

all_paths = au_paths + tp_paths
all_labels = [0] * len(au_paths) + [1] * len(tp_paths)

indices = list(range(len(all_paths)))
idx_tv, idx_test = train_test_split(indices, test_size=0.15, stratify=all_labels, random_state=SEED)
labels_tv = [all_labels[i] for i in idx_tv]
idx_train, idx_val = train_test_split(idx_tv, test_size=0.15 / 0.85, stratify=labels_tv, random_state=SEED)

train_paths = [all_paths[i] for i in idx_train]
train_labels = [all_labels[i] for i in idx_train]
val_paths = [all_paths[i] for i in idx_val]
val_labels = [all_labels[i] for i in idx_val]
test_paths = [all_paths[i] for i in idx_test]
test_labels = [all_labels[i] for i in idx_test]

for name, lbl in [("Train", train_labels), ("Val", val_labels), ("Test", test_labels)]:
    print(f"{name:5s}: total={len(lbl):4d}  Au={lbl.count(0):4d}  Tp={lbl.count(1):4d}")

def source_tokens_from_filename(path_obj: Path) -> set[str]:
    """Extract source-like tokens from a CASIA filename."""
    stem = path_obj.stem.lower()
    stem = re.sub(r"^(au|tp)[_\- ]*", "", stem)

    tokens = set()

    # Matches forms such as arc_00013, arc00013, sec00045, etc.
    for letters, digits in re.findall(r"([a-z]+)[_\- ]*(\d+)", stem):
        tokens.add(f"{letters}{digits}")

    # Catch already compact fragments if they appear directly.
    tokens.update(re.findall(r"[a-z]+\d+", stem.replace("_", "").replace("-", "")))

    # Conservative fallback if nothing obvious was found.
    if not tokens:
        compact = re.sub(r"[^a-z0-9]+", "", stem)
        if compact:
            tokens.add(compact)

    return tokens

# Build the set of authentic-source tokens present in the TRAIN split.
train_auth_tokens = set()
for p in train_paths:
    if p.parent.name == "Au":
        train_auth_tokens.update(source_tokens_from_filename(p))

# Only tampered TEST queries are relevant for the leakage audit.
tp_test_paths = [p for p, y in zip(test_paths, test_labels) if y == 1]

leaked = []
for p in tp_test_paths:
    src_tokens = source_tokens_from_filename(p)
    shared = sorted(src_tokens & train_auth_tokens)
    if shared:
        leaked.append((p.name, shared))

print()
print(f"Tampered test queries          : {len(tp_test_paths):4d}")
print(f"Authentic source tokens in train: {len(train_auth_tokens):4d}")
print(f"Queries with source in train   : {len(leaked):4d}")
print(f"Leakage rate                   : {100 * len(leaked) / len(tp_test_paths):5.1f}%")

if leaked:
    print()
    print("Examples:")
    for name, shared in leaked[:10]:
        print(f"  {name} -> {', '.join(shared)}")


100%|██████████| 2.56G/2.56G [00:27<00:00, 98.2MB/s]

Extracting files...


Train: total=8829  Au=5243  Tp=3586
Val  : total=1892  Au=1124  Tp= 768
Test : total=1893  Au=1124  Tp= 769

Tampered test queries          :  769
Authentic source tokens in train: 5243
Queries with source in train   :  600
Leakage rate                   :  78.0%

Examples:
  Tp_S_NNN_S_N_pla00038_pla00038_00573.tif -> pla00038
  Tp_D_NRN_S_N_cha00008_art10102_11163.jpg -> cha00008
  Tp_D_NRN_S_N_arc00013_arc00020_11702.jpg -> arc00013, arc00020
  Tp_S_NNN_S_N_cha20004_cha20004_02003.tif -> cha20004
  Tp_S_NNN_S_N_cha10202_cha10202_12356.jpg -> cha10202
  Tp_S_NRN_S_B_txt00022_txt00022_20089.jpg -> txt00022
  Tp_S_NNN_S_B_sec20060_sec20060_02148.tif -> sec20060
  Tp_S_NNN_S_N_pla20067_pla20067_02367.tif -> pla20067
  Tp_D_NRN_M_O_nat10151_cha00062_12107.jpg -> cha00062, nat10151
  Tp_S_NNN_S_N_nat20060_nat20060_02254.tif -> nat20060
